# 🏋️ Trading Environment Testing

This notebook tests the custom OpenAI Gym trading environment with real Bitcoin price data.

**Project:** DeepTrade-RL - A Reinforcement Learning Bitcoin Trading Agent  
**Course:** SE4050 Deep Learning

## Objectives:
1. Load preprocessed Bitcoin price data
2. Initialize the trading environment
3. Test environment with random actions
4. Analyze trading performance
5. Verify environment correctness

---

In [ ]:
# Install required packages (Google Colab)
!pip install -q gym numpy pandas matplotlib

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Import Libraries and Load Data

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gym
from gym import spaces
from typing import Dict, Tuple

# Set up project path
PROJECT_PATH = "/content/drive/MyDrive/deeptrade-rl"
sys.path.append(PROJECT_PATH)

# Load cached market data
data_file = f"{PROJECT_PATH}/data/cached_market_data.csv"
df = pd.read_csv(data_file, parse_dates=['timestamp'])

print(f"Loaded {len(df)} rows of market data")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nColumns: {list(df.columns)}")

## Define Trading Environment

We'll implement a simplified version of the trading environment inline for testing.

In [ ]:
class TradingEnvironment(gym.Env):
    """
    Custom OpenAI Gym environment for cryptocurrency trading.
    
    Action Space:
        - 0: HOLD (do nothing)
        - 1: BUY (buy BTC with available cash)
        - 2: SELL (sell all BTC holdings)
    """
    
    def __init__(self, df, initial_balance=10000.0, transaction_fee=0.001, lookback_window=50):
        super().__init__()
        
        self.df = df.reset_index(drop=True)
        self.initial_balance = initial_balance
        self.transaction_fee = transaction_fee
        self.lookback_window = lookback_window
        
        # Action space: HOLD, BUY, SELL
        self.action_space = spaces.Discrete(3)
        
        # State dimension: portfolio (3) + indicators (5) + price history (50)
        self.state_dim = 3 + 5 + lookback_window
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.state_dim,), dtype=np.float32
        )
        
        # Episode tracking
        self.current_step = 0
        self.max_steps = len(self.df) - lookback_window - 1
        
        # Portfolio state
        self.cash = initial_balance
        self.btc_holdings = 0.0
        self.portfolio_values = []
        self.trades = []
        
    def reset(self):
        """Reset environment to initial state."""
        self.current_step = 0
        self.cash = self.initial_balance
        self.btc_holdings = 0.0
        self.portfolio_values = [self.initial_balance]
        self.trades = []
        return self._get_observation()
    
    def step(self, action):
        """Execute one trading step."""
        if self.current_step >= self.max_steps:
            raise ValueError("Episode finished. Reset environment.")
        
        # Get current price
        current_price = self._get_current_price()
        portfolio_before = self._get_portfolio_value()
        
        # Execute action
        transaction_cost = 0.0
        
        if action == 1 and self.cash > 0:  # BUY
            btc_to_buy = self.cash / current_price
            transaction_cost = btc_to_buy * current_price * self.transaction_fee
            btc_after_fee = btc_to_buy - (transaction_cost / current_price)
            
            self.btc_holdings += btc_after_fee
            self.cash = 0.0
            self.trades.append(("BUY", self.current_step, current_price, btc_after_fee))
            
        elif action == 2 and self.btc_holdings > 0:  # SELL
            cash_from_sale = self.btc_holdings * current_price
            transaction_cost = cash_from_sale * self.transaction_fee
            
            self.cash += cash_from_sale - transaction_cost
            self.btc_holdings = 0.0
            self.trades.append(("SELL", self.current_step, current_price, 0))
        
        # Move forward
        self.current_step += 1
        
        # Calculate reward
        portfolio_after = self._get_portfolio_value()
        reward = (portfolio_after - portfolio_before) / self.initial_balance
        reward -= (transaction_cost / self.initial_balance)
        
        self.portfolio_values.append(portfolio_after)
        
        # Check if done
        done = self.current_step >= self.max_steps
        
        # Get observation
        observation = self._get_observation()
        
        info = {
            "step": self.current_step,
            "portfolio_value": portfolio_after,
            "cash": self.cash,
            "btc_holdings": self.btc_holdings
        }
        
        return observation, reward, done, info
    
    def _get_observation(self):
        """Construct state observation."""
        idx = self.lookback_window + self.current_step
        current_price = self.df.loc[idx, "close"]
        
        # Portfolio state
        state = [
            self.cash / self.initial_balance,
            self.btc_holdings * current_price / self.initial_balance,
            current_price / 10000.0
        ]
        
        # Technical indicators (simplified)
        state.extend([
            self.df.loc[idx, "rsi"] / 100.0 if "rsi" in self.df.columns else 0.5,
            0.0,  # Placeholder for other indicators
            0.0,
            0.0,
            0.0
        ])
        
        # Price history
        price_history = self.df.loc[idx - self.lookback_window:idx, "close"].values
        price_history_norm = price_history / price_history[0]
        state.extend(price_history_norm.tolist())
        
        return np.array(state, dtype=np.float32)
    
    def _get_current_price(self):
        """Get current BTC price."""
        idx = self.lookback_window + self.current_step
        return self.df.loc[idx, "close"]
    
    def _get_portfolio_value(self):
        """Calculate total portfolio value."""
        return self.cash + (self.btc_holdings * self._get_current_price())
    
    def get_performance_metrics(self):
        """Calculate performance metrics."""
        portfolio_values = np.array(self.portfolio_values)
        final_value = portfolio_values[-1]
        total_return = (final_value - self.initial_balance) / self.initial_balance
        
        # Calculate returns
        returns = np.diff(portfolio_values) / portfolio_values[:-1]
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252 * 24) if returns.std() > 0 else 0.0
        
        # Max drawdown
        cumulative = portfolio_values / portfolio_values[0]
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        
        return {
            "initial_balance": self.initial_balance,
            "final_portfolio_value": final_value,
            "total_return": total_return,
            "total_return_pct": total_return * 100,
            "sharpe_ratio": sharpe_ratio,
            "max_drawdown": max_drawdown,
            "max_drawdown_pct": max_drawdown * 100,
            "total_trades": len(self.trades)
        }

print("Trading environment defined")

## Test Environment with Random Actions

In [ ]:
# Initialize environment
env = TradingEnvironment(df, initial_balance=10000, transaction_fee=0.001)

print(f"Environment initialized:")
print(f"  State dimension: {env.state_dim}")
print(f"  Action space: {env.action_space}")
print(f"  Max steps: {env.max_steps}")

# Test reset
state = env.reset()
print(f"\nReset successful. State shape: {state.shape}")
print(f"Initial observation (first 10 values): {state[:10]}")

## Run Episode with Random Actions

In [ ]:
# Run one episode with random actions
state = env.reset()
episode_rewards = []
actions_taken = []

print("Running episode with random actions...")

for step in range(env.max_steps):
    # Random action
    action = env.action_space.sample()
    
    # Take step
    next_state, reward, done, info = env.step(action)
    
    episode_rewards.append(reward)
    actions_taken.append(action)
    
    # Log every 500 steps
    if (step + 1) % 500 == 0:
        print(f"  Step {step + 1}/{env.max_steps}: "
              f"Action={action}, "
              f"Reward={reward:.6f}, "
              f"Portfolio=${info['portfolio_value']:,.2f}")
    
    if done:
        break
    
    state = next_state

print(f"\nEpisode complete!")
print(f"Total steps: {len(episode_rewards)}")
print(f"Total rewards: {sum(episode_rewards):.4f}")

## Analyze Performance

In [ ]:
# Get performance metrics
metrics = env.get_performance_metrics()

print("Data Performance Metrics:")
print(f"  Initial Balance: ${metrics['initial_balance']:,.2f}")
print(f"  Final Portfolio Value: ${metrics['final_portfolio_value']:,.2f}")
print(f"  Total Return: {metrics['total_return_pct']:.2f}%")
print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
print(f"  Max Drawdown: {metrics['max_drawdown_pct']:.2f}%")
print(f"  Total Trades: {metrics['total_trades']}")

# Action distribution
action_names = ["HOLD", "BUY", "SELL"]
action_counts = [actions_taken.count(i) for i in range(3)]

print(f"\nChart Action Distribution:")
for i, name in enumerate(action_names):
    pct = (action_counts[i] / len(actions_taken)) * 100
    print(f"  {name}: {action_counts[i]} ({pct:.1f}%)")

## Visualize Results

In [ ]:
# Plot portfolio value over time
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Portfolio value
axes[0].plot(env.portfolio_values, linewidth=1, color='blue')
axes[0].axhline(y=env.initial_balance, color='r', linestyle='--', alpha=0.5, label='Initial Balance')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Portfolio Value Over Time (Random Actions)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative rewards
cumulative_rewards = np.cumsum(episode_rewards)
axes[1].plot(cumulative_rewards, linewidth=1, color='green')
axes[1].set_ylabel('Cumulative Reward')
axes[1].set_title('Cumulative Rewards')
axes[1].grid(True, alpha=0.3)

# Actions taken
colors = ['gray', 'green', 'red']
axes[2].scatter(range(len(actions_taken)), actions_taken, 
                c=[colors[a] for a in actions_taken], s=1, alpha=0.5)
axes[2].set_ylabel('Action')
axes[2].set_xlabel('Step')
axes[2].set_title('Actions Taken (0=HOLD, 1=BUY, 2=SELL)')
axes[2].set_yticks([0, 1, 2])
axes[2].set_yticklabels(action_names)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PROJECT_PATH}/data/environment_test_results.png", dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved!")

## Test with Buy-and-Hold Strategy (Baseline)

In [ ]:
# Run buy-and-hold strategy for comparison
env_bh = TradingEnvironment(df, initial_balance=10000, transaction_fee=0.001)
state_bh = env_bh.reset()

# Buy at the start
state_bh, reward_bh, done_bh, info_bh = env_bh.step(1)  # BUY

# Hold for the rest
for step in range(env_bh.max_steps - 1):
    state_bh, reward_bh, done_bh, info_bh = env_bh.step(0)  # HOLD
    if done_bh:
        break

# Sell at the end
if not done_bh:
    state_bh, reward_bh, done_bh, info_bh = env_bh.step(2)  # SELL

metrics_bh = env_bh.get_performance_metrics()

print("Data Buy-and-Hold Strategy (Baseline):")
print(f"  Final Portfolio Value: ${metrics_bh['final_portfolio_value']:,.2f}")
print(f"  Total Return: {metrics_bh['total_return_pct']:.2f}%")
print(f"  Total Trades: {metrics_bh['total_trades']}")

# Compare
print(f"\n🔍 Comparison:")
print(f"  Random Actions Return: {metrics['total_return_pct']:.2f}%")
print(f"  Buy-and-Hold Return: {metrics_bh['total_return_pct']:.2f}%")
print(f"  Difference: {metrics['total_return_pct'] - metrics_bh['total_return_pct']:.2f}%")

## Environment Testing Complete!

**Summary:**
- Trading environment successfully initialized
- State and action spaces verified
- Random action episode completed
- Performance metrics calculated
- Buy-and-hold baseline established

**Key Findings:**
- Environment properly handles BUY, SELL, and HOLD actions
- Portfolio value and rewards are tracked correctly
- State observations include price history and indicators
- Ready for RL agent training!

**Next Steps:**
- Notebook 03: Train DDQN agent on this environment
- Notebook 04: Train PPO agent
- Compare agent performance vs. baselines

---